# Arlington County Traffic Crash Analysis (2019–2023)

**Author:** Munkhtenger Enkhbold  
**Data Source:** Arlington County Open Data Portal (simulated for analysis practice)  
**Purpose:** Exploratory data analysis to identify crash patterns, high-risk locations, and vulnerable road user trends in support of traffic safety planning.

---

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="Blues_d")
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

df = pd.read_csv("arlington_crashes_2019_2023.csv")
df["date"] = pd.to_datetime(df["date"])

print(f"Dataset loaded: {len(df):,} crash records")
print(f"Columns: {list(df.columns)}")
df.head()

## 1. Overview & Summary Statistics

In [ ]:
print("=== Dataset Summary ===")
print(f"Total crashes: {len(df):,}")
print(f"Years: {df['year'].min()} to {df['year'].max()}")
print(f"Unique streets: {df['street'].nunique()}")
print()
print("Severity breakdown:")
print(df["severity"].value_counts())
print()
print("Crash types:")
print(df["crash_type"].value_counts())

## 2. Crash Trends Over Time

Understanding year-over-year trends helps assess whether safety interventions are working.

In [ ]:
yearly = df.groupby("year").size().reset_index(name="crashes")

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(yearly["year"], yearly["crashes"], color="#1F4E79", width=0.6, zorder=3)
ax.set_title("Total Crashes by Year — Arlington County", fontsize=14, fontweight="bold", pad=15)
ax.set_xlabel("Year")
ax.set_ylabel("Number of Crashes")
ax.set_xticks(yearly["year"])
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            str(int(bar.get_height())), ha="center", fontsize=10)
ax.grid(axis="y", alpha=0.4, zorder=0)
plt.tight_layout()
plt.show()

## 3. Time-of-Day Patterns

Identifying peak crash hours helps prioritize enforcement and infrastructure improvements.

In [ ]:
hourly = df.groupby("hour").size().reset_index(name="crashes")

fig, ax = plt.subplots(figsize=(11, 5))
ax.fill_between(hourly["hour"], hourly["crashes"], alpha=0.2, color="#2E75B6")
ax.plot(hourly["hour"], hourly["crashes"], color="#1F4E79", linewidth=2.5, marker="o", markersize=4)
ax.axvspan(7, 9, alpha=0.08, color="red", label="Morning rush (7-9am)")
ax.axvspan(16, 19, alpha=0.08, color="orange", label="Evening rush (4-7pm)")
ax.set_title("Crash Frequency by Hour of Day", fontsize=14, fontweight="bold", pad=15)
ax.set_xlabel("Hour of Day (24h)")
ax.set_ylabel("Number of Crashes")
ax.set_xticks(range(0, 24))
ax.legend()
ax.grid(axis="y", alpha=0.4)
plt.tight_layout()
plt.show()

peak = hourly.loc[hourly["crashes"].idxmax()]
print(f"Peak crash hour: {int(peak['hour'])}:00 with {int(peak['crashes'])} crashes")

## 4. High-Risk Locations

Identifying crash hotspots by street supports targeted safety improvements.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
street_counts = df["street"].value_counts().head(10).sort_values()
colors = ["#1F4E79" if i >= len(street_counts) - 3 else "#AEC6E8" for i in range(len(street_counts))]
ax.barh(street_counts.index, street_counts.values, color=colors, height=0.6)
ax.set_title("Top 10 Streets by Crash Frequency", fontsize=14, fontweight="bold", pad=15)
ax.set_xlabel("Number of Crashes")
for i, v in enumerate(street_counts.values):
    ax.text(v + 1, i, str(v), va="center", fontsize=10)
ax.grid(axis="x", alpha=0.4)
plt.tight_layout()
plt.show()

## 5. Severity Analysis

In [ ]:
sev_order = ["Property Damage Only", "Injury", "Serious Injury", "Fatal"]
sev_counts = df["severity"].value_counts().reindex(sev_order)
colors = ["#AEC6E8", "#2E75B6", "#1F4E79", "#8B0000"]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(sev_counts.index, sev_counts.values, color=colors, width=0.55, zorder=3)
ax.set_title("Crash Severity Distribution", fontsize=14, fontweight="bold", pad=15)
ax.set_ylabel("Number of Crashes")
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f"{int(bar.get_height())}\n({int(bar.get_height())/len(df)*100:.1f}%)",
            ha="center", va="bottom", fontsize=9.5)
ax.grid(axis="y", alpha=0.4, zorder=0)
plt.tight_layout()
plt.show()

## 6. Vulnerable Road Users — Pedestrians & Cyclists

Vision Zero prioritizes protecting the most vulnerable road users. This section identifies which streets have the highest pedestrian and cyclist involvement.

In [ ]:
vuln = df.groupby("street")[["pedestrians_involved", "cyclists_involved"]].sum().sort_values("pedestrians_involved", ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(vuln))
width = 0.35
ax.bar([i - width/2 for i in x], vuln["pedestrians_involved"], width, label="Pedestrians", color="#1F4E79")
ax.bar([i + width/2 for i in x], vuln["cyclists_involved"], width, label="Cyclists", color="#AEC6E8")
ax.set_title("Pedestrian & Cyclist Involvement by Street", fontsize=14, fontweight="bold", pad=15)
ax.set_ylabel("Count")
ax.set_xticks(list(x))
ax.set_xticklabels(vuln.index, rotation=30, ha="right", fontsize=9)
ax.legend()
ax.grid(axis="y", alpha=0.4)
plt.tight_layout()
plt.show()

print(f"Total pedestrian crashes: {df['pedestrians_involved'].sum()}")
print(f"Total cyclist crashes: {df['cyclists_involved'].sum()}")
print(f"\nTop street for pedestrian crashes: {vuln['pedestrians_involved'].idxmax()}")
print(f"Top street for cyclist crashes: {vuln['cyclists_involved'].idxmax()}")

## 7. Key Findings & Recommendations

Based on this analysis:

1. **Peak crash hours are 5pm–6pm** — evening rush hour is the highest risk window. Targeted enforcement and signal timing adjustments during this period could reduce crashes.

2. **Top 3 high-risk streets** require priority attention for infrastructure review, signage, and speed management.

3. **Pedestrian and cyclist crashes are concentrated on specific corridors** — these should be cross-referenced with existing Vision Zero hotspot maps for intervention prioritization.

4. **Serious injury and fatal crashes account for ~9% of all crashes** — while relatively low, these are the cases Vision Zero aims to eliminate entirely by 2030.

5. **Year-over-year trend monitoring** should continue to evaluate the effectiveness of safety countermeasures as they are implemented.

---
*This analysis was conducted as part of a transportation safety data project. Data represents a sample dataset structured to reflect realistic Arlington County crash patterns.*